In [1]:
# [Colab Cell 1] Setup
# Update transformers to the latest dev version for Qwen2.5-VL support
!pip install -q "numpy<2.0.0"
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q vllm==0.6.3 pillow tqdm torch torchvision qwen-vl-utils decord datasets

import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU not detected.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
GPU: Tesla T4


In [2]:
# [Colab Cell 2] Clone and explore
import os
%cd /content/
if os.path.exists('DisasterM3'):
    !rm -rf DisasterM3

!git clone https://github.com/Junjue-Wang/DisasterM3.git
%cd DisasterM3

# Show key files using the correct path
!ls -la pyscripts/

/content
Cloning into 'DisasterM3'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 82 (delta 24), reused 24 (delta 24), pack-reused 56 (from 1)
Receiving objects: 100% (82/82), 22.22 KiB | 1.85 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/DisasterM3
total 24
drwxr-xr-x 2 root root  4096 May 30 07:58 .
drwxr-xr-x 5 root root  4096 May 30 07:58 ..
-rw-r--r-- 1 root root     0 May 30 07:58 __init__.py
-rw-r--r-- 1 root root 12432 May 30 07:58 run_vllm.py


In [3]:
from datasets import load_dataset
from google.colab import userdata
import os
from huggingface_hub import login

# 1. Handle Authentication using the secret you just created
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("✓ Successfully authenticated with Hugging Face token!")
except Exception as e:
    print("⚠️ HF_TOKEN not found in Secrets. Please add it via the 🔑 icon and enable 'Notebook access'.")

# 2. Load the actual dataset (replacing the mock data)
ds = None
try:
    print("Attempting to load real dataset samples...")
    # Using streaming=True to bypass large file download overhead
    stream_ds = load_dataset("Kingdrone-Junjue/DisasterM3", split="test", streaming=True)
    ds = list(stream_ds.take(5))
    print(f"✓ Success! Loaded {len(ds)} real samples from Hugging Face.")
    print(f"Available fields: {list(ds[0].keys())}")
except Exception as e:
    print(f"✗ Load failed: {e}")
    print("Check your token permissions or wait a few minutes for the rate limit to reset.")

✓ Successfully authenticated with Hugging Face token!
Attempting to load real dataset samples...


Resolving data files:   0%|          | 0/22059 [00:00<?, ?it/s]

✓ Success! Loaded 5 real samples from Hugging Face.
Available fields: ['image']


In [11]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "pyscripts"))
from run_vllm import prompt_libs

# Manually construct messages to use PIL objects directly instead of file paths
def get_manual_messages(sample, subset="bearing_body"):
    prompt = prompt_libs[subset].replace("{prompts}", sample['prompts']).replace("{options_str}", sample['options_str'])
    # Split the prompt into segments based on where images should go
    # Note: Qwen2.5-VL processor expects a list of dicts with 'image' as a PIL object or URL
    messages = [{
        "role": "user",
        "content": [
            {"type": "text", "text": "Analyze both the pre-disaster and post-disaster images to answer the following question. Choose the best option(s) from the candidate options provided.\n\npre-disaster image:"},
            {"type": "image", "image": sample['pre_image']},
            {"type": "text", "text": "\n\npost-disaster image:"},
            {"type": "image", "image": sample['post_image']},
            {"type": "text", "text": f"\n\nQuestion: {sample['prompts']}\nOptions: {sample['options_str']}\n\nYour task is to respond with ONLY the capital letters of the correct options."}
        ]
    }]
    return messages

# Create a test sample using actual PIL images from the dataset
test_sample = {
    'pre_image': ds[0]['image'],
    'post_image': ds[0]['image'],
    'prompts': 'Is there any building damage visible?',
    'options_str': 'A: Yes, B: No',
    'answer': 'B'
}

messages = get_manual_messages(test_sample)
print("✓ Messages constructed with PIL objects directly!")

✓ Messages constructed with PIL objects directly!


In [5]:
# [Colab Cell 5] Load Qwen2.5-VL-3B-Instruct
# REQUIRED: You MUST Restart session (Runtime > Restart session) before running this.

try:
    import torch
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

    model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
    print(f"Loading {model_id} via Transformers Dev Branch...")

    # Use float16 for T4 GPU compatibility and memory efficiency
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    print("\n✓ Model and Processor loaded successfully!")
except ImportError:
    print("\n❌ Architecture not found. Please Restart Session and run Cell 1 Setup again.")
except Exception as e:
    print(f"\n❌ Loading failed: {e}")

Loading Qwen/Qwen2.5-VL-3B-Instruct via Transformers Dev Branch...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]


✓ Model and Processor loaded successfully!


In [8]:
# [Colab Cell 6] Create the deliverable file
execution_notes = """# Execution Notes: DisasterM3 Benchmark Script

**Date**: 2024-05-30
**Environment**: Google Colab (T4 GPU, 16GB VRAM)
**Dataset**: Kingdrone-Junjue/DisasterM3 (Hugging Face)
**Script**: disaster_m3/pyscripts/run_vllm.py

## Dependencies Installed
- transformers (dev branch)
- vllm==0.6.3
- torch
- datasets
"""

with open("execution_notes.md", "w") as f:
    f.write(execution_notes)

print("✓ execution_notes.md has been created.")

✓ execution_notes.md has been created.


In [12]:
try:
    from qwen_vl_utils import process_vision_info

    # 1. Apply the chat template
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # 2. Extract images/videos from the message content
    # Since we passed PIL objects in the content list, process_vision_info will handle them correctly
    image_inputs, video_inputs = process_vision_info(messages)

    # 3. Prepare inputs for the model
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    # 4. Generate Output
    print("Running inference...")
    generated_ids = model.generate(**inputs, max_new_tokens=128)
    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )

    print("\n--- Model Result ---")
    print(f"Answer: {output_text[0]}")
    print(f"Ground Truth: {test_sample['answer']}")

except Exception as e:
    print(f"❌ Inference failed: {e}")
    import traceback
    traceback.print_exc()

Running inference...

--- Model Result ---
Answer: B
Ground Truth: B
